In [8]:
# %%
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg') # Grafiklerin ekrana basilmasini engeller (sunucular icin iyi)
import matplotlib.pyplot as plt
import joblib
import shap
import lime
import lime.lime_tabular
import tensorflow as tf
import warnings
import glob
import scipy.ndimage
from sklearn.impute import SimpleImputer
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization, 
    GlobalAveragePooling1D, Conv1D, ReLU, Add, AdditiveAttention
)

warnings.filterwarnings('ignore')

In [9]:
BASE_PATH = '/media/data/bitirme son/bitirme-projesi'
PROCESSED_DATA_PATH = os.path.join(BASE_PATH, 'data', 'processed')
MODELS_PATH = os.path.join(BASE_PATH, 'models')
ENSEMBLE_PATH = os.path.join(MODELS_PATH, 'ensemble')
ML_GAN_MODELS_PATH = os.path.join(MODELS_PATH, 'gan_balanced')
DL_GAN_MODELS_PATH = os.path.join(MODELS_PATH, 'dl_gan_balanced')
ORIG_SIGNALS_PATH = os.path.join(PROCESSED_DATA_PATH, 'signals')
RESULTS_PATH = os.path.join(BASE_PATH, 'results')
os.makedirs(RESULTS_PATH, exist_ok=True)


CLASS_NAMES = ['CD', 'HYP', 'MI', 'NORM', 'STTC']

In [10]:
print("--- VERİ HAZIRLIĞI ---")

X_test_feat = np.load(os.path.join(PROCESSED_DATA_PATH, 'X_test_feat.npy'))
X_test_sig = np.load(os.path.join(ORIG_SIGNALS_PATH, 'X_test_sig.npy'))
y_test = np.load(os.path.join(PROCESSED_DATA_PATH, 'y_test.npy'))


scaler_path = os.path.join(MODELS_PATH, 'scaler.pkl')
if not os.path.exists(scaler_path):
     scaler_path = os.path.join(ML_GAN_MODELS_PATH, 'scaler_balanced.pkl')
scaler = joblib.load(scaler_path)


imputer = SimpleImputer(strategy='mean')
X_test_imputed = imputer.fit_transform(X_test_feat)
X_test_scaled = scaler.transform(X_test_imputed)

--- VERİ HAZIRLIĞI ---


In [11]:
# --- MODEL YÜKLEME ---
# 1. Stacking Modeli (ML Ensemble - SHAP ve LIME icin)
stacking_model = joblib.load(os.path.join(ENSEMBLE_PATH, 'ml_stacking_proper_model.pkl'))
print("Stacking modeli yüklendi.")

Stacking modeli yüklendi.


In [12]:
# 2. ResNet Modeli (DL - Grad-CAM icin)
def resnet_block(inputs, filters, kernel, stride):
    shortcut = inputs
    x = Conv1D(filters, kernel, strides=stride, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv1D(filters, kernel, padding='same')(x)
    x = BatchNormalization()(x)
    
    if stride != 1 or inputs.shape[-1] != filters:
         shortcut = Conv1D(filters, 1, strides=stride, padding='same')(inputs)
         shortcut = BatchNormalization()(shortcut)
    
    x = Add()([x, shortcut])
    x = ReLU()(x)
    return x

def build_resnet_architecture(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = Conv1D(64, 7, strides=2, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    
    x = resnet_block(x, 64, 3, 1)
    x = resnet_block(x, 64, 3, 1)
    x = resnet_block(x, 128, 3, 2)
    x = resnet_block(x, 128, 3, 1)
    x = resnet_block(x, 256, 3, 2)
    x = resnet_block(x, 256, 3, 1)
    
    attn = AdditiveAttention(name='attention_layer')([x, x])
    x = GlobalAveragePooling1D()(attn)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    return Model(inputs, outputs)

time_steps = X_test_sig.shape[1]
n_leads = X_test_sig.shape[2]
resnet_model = build_resnet_architecture((time_steps, n_leads), 5)

In [13]:
# Agirliklari bul ve yukle
weight_search_pattern = os.path.join(DL_GAN_MODELS_PATH, '*resnet*epoch*50*.h5')
found_files = glob.glob(weight_search_pattern)
if not found_files:
    weight_search_pattern_backup = os.path.join(DL_GAN_MODELS_PATH, 'checkpoint*weights.h5')
    found_files = glob.glob(weight_search_pattern_backup)
    
if found_files:
    resnet_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    try:
        resnet_model.load_weights(found_files[0], skip_mismatch=True)
        print(f"ResNet ağirliklari yüklendi: {os.path.basename(found_files[0])}")
    except Exception as e:
        print(f"ResNet yükleme hatasi: {e}")
else:
    print("ResNet ağirlik dosyasi bulunamadi! Grad-CAM çalişmayabilir.")

ResNet ağirliklari yüklendi: checkpoint_resnet34_with_attention_gan_epoch_50.weights.h5


In [14]:
# --- ÖZELLİK İSİMLERİNİ OLUŞTURMA ---
def generate_feature_names(n_leads=12, n_stat=8, n_freq=3, n_hrv=15):
    leads = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
    if len(leads) != n_leads:
        leads = [f'Lead_{i+1}' for i in range(n_leads)]
    
    feature_names = []
    stat_suffixes = ['Mean', 'Std', 'Skew', 'Kurtosis', 'RMS', 'Entropy', 'Mean_Abs_Diff', 'Zero_Crossings']
    for lead in leads:
        feature_names.extend([f"{lead}_{suffix}" for suffix in stat_suffixes])
            
    freq_suffixes = ['PSD_Mean', 'PSD_Std', 'Spectral_Entropy']
    for lead in leads:
        feature_names.extend([f"{lead}_{suffix}" for suffix in freq_suffixes])
            
    hrv_count = n_hrv - 4
    feature_names.extend([f"HRV_Param_{i+1}" for i in range(hrv_count)])
    feature_names.extend(['QRS_Duration', 'R_Amplitude', 'T_Amplitude', 'P_Amplitude'])
    
    return feature_names

FEATURE_NAMES = generate_feature_names(n_leads=X_test_sig.shape[2])

In [15]:
# 1. SHAP ANALİZİ (Summary Plot + Feature Importance Bar Plot)
# =============================================================================
print("\n--- SHAP ANALİZİ BAŞLATILIYOR ---")
SHAP_FILE_PATH = os.path.join(BASE_PATH, 'shap_values_global.pkl')

nsamples_shap = 2000
np.random.seed(42)
random_indices = np.random.choice(X_test_scaled.shape[0], nsamples_shap, replace=False)
X_shap_explain = X_test_scaled[random_indices]

if os.path.exists(SHAP_FILE_PATH):
    print("   -> Kayitli SHAP değerleri bulundu, yükleniyor...")
    shap_values = joblib.load(SHAP_FILE_PATH)
else:
    print("   -> SHAP değerleri hesaplaniyor (K-Means özeti ile)...")
    summary_k = 20
    X_summary = shap.kmeans(X_test_scaled, summary_k)
    explainer = shap.KernelExplainer(stacking_model.predict_proba, X_summary)
    shap_values = explainer.shap_values(X_shap_explain, nsamples=100)
    joblib.dump(shap_values, SHAP_FILE_PATH)
    print("   -> SHAP değerleri hesaplandi ve kaydedildi.")

# --- A) SHAP Summary Plot (Beeswarm) ---
plt.figure(figsize=(14, 10))
shap.summary_plot(shap_values, X_shap_explain, feature_names=FEATURE_NAMES, class_names=CLASS_NAMES, show=False)
shap_img_path = os.path.join(RESULTS_PATH, 'shap_summary_plot.png')
plt.savefig(shap_img_path, bbox_inches='tight', dpi=300)
plt.close()
print(f"SHAP Summary Plot kaydedildi: {shap_img_path}")

# --- B) SHAP Feature Importance (Bar Plot) ---
print("   -> SHAP Özellik Önem Grafiği (Bar Plot) çiziliyor...")
plt.figure(figsize=(10, 8))
# 'plot_type="bar"' parametresi ile çubuk grafik çizdiriyoruz.
# shap_values bir liste olduğu için (her sınıf için bir matris), tüm sınıflar üzerindeki ortalama mutlak değeri gösterir.
shap.summary_plot(shap_values, X_shap_explain, feature_names=FEATURE_NAMES, class_names=CLASS_NAMES, plot_type="bar", max_display=10, show=False)
plt.title("SHAP Özellik Önemi (Global)", fontsize=16, fontweight='bold')
shap_bar_path = os.path.join(RESULTS_PATH, 'shap_feature_importance_bar.png')
plt.savefig(shap_bar_path, bbox_inches='tight', dpi=300)
plt.close()
print(f"SHAP Özellik Önem Grafiği kaydedildi: {shap_bar_path}")



--- SHAP ANALİZİ BAŞLATILIYOR ---
   -> SHAP değerleri hesaplaniyor (K-Means özeti ile)...


  0%|          | 0/2000 [00:00<?, ?it/s]

   -> SHAP değerleri hesaplandi ve kaydedildi.
SHAP Summary Plot kaydedildi: /media/data/bitirme son/bitirme-projesi/results/shap_summary_plot.png
   -> SHAP Özellik Önem Grafiği (Bar Plot) çiziliyor...
SHAP Özellik Önem Grafiği kaydedildi: /media/data/bitirme son/bitirme-projesi/results/shap_feature_importance_bar.png


In [16]:
# 2. LIME ANALİZİ (Detayli Hasta Analizi)
# =============================================================================
print("\n--- LIME ANALİZİ BAŞLATILIYOR ---")
lime_background_size = 2000
bg_idx = np.random.choice(X_test_scaled.shape[0], lime_background_size, replace=False)

explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_test_scaled[bg_idx],
    feature_names=FEATURE_NAMES,
    class_names=CLASS_NAMES,
    mode='classification',
    discretize_continuous=True # Feature degerlerini araliklara boler (orneginizdeki gibi)
)

# Rastgele bir hasta seç
patient_idx = np.random.randint(0, X_test_scaled.shape[0])
patient_data_scaled = X_test_scaled[patient_idx]
patient_data_original = X_test_imputed[patient_idx] # Orijinal (scale edilmemiş ama doldurulmuş) değerler

print(f"   -> Analiz edilen hasta indeksi: {patient_idx}")

# Modelin bu hasta için tahmin olasılıkları
probs = stacking_model.predict_proba(patient_data_scaled.reshape(1, -1))[0]
pred_label_idx = np.argmax(probs)
pred_class_str = CLASS_NAMES[pred_label_idx]

# --- A) LIME Açıklama Grafiği ---
exp = explainer_lime.explain_instance(
    data_row=patient_data_scaled, 
    predict_fn=stacking_model.predict_proba,
    num_features=10, 
    top_labels=1 
)

fig_lime = exp.as_pyplot_figure(label=pred_label_idx)
plt.title(f"LIME Analizi (Hasta: {patient_idx}, Tahmin: {pred_class_str})")
plt.tight_layout()
lime_plot_path = os.path.join(RESULTS_PATH, f'lime_analizi_hasta_{patient_idx}.png')
fig_lime.savefig(lime_plot_path, bbox_inches='tight', dpi=150)
plt.close(fig_lime)
print(f"LIME grafiği kaydedildi: {lime_plot_path}")

# --- B) Tahmin Olasılıkları Tablosu (Resim olarak kaydetme) ---
print("   -> Tahmin olasılıkları tablosu oluşturuluyor...")
probs_df = pd.DataFrame({'Sınıf': CLASS_NAMES, 'Olasılık (%)': (probs * 100).round(2)})
probs_df = probs_df.sort_values(by='Olasılık (%)', ascending=False)

fig, ax = plt.subplots(figsize=(6, 4))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=probs_df.values, colLabels=probs_df.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.2)
plt.title(f"Hasta {patient_idx} İçin Model Tahmin Olasılıkları", fontsize=14, fontweight='bold')
probs_table_path = os.path.join(RESULTS_PATH, f'lime_probs_table_hasta_{patient_idx}.png')
plt.savefig(probs_table_path, bbox_inches='tight', dpi=150)
plt.close()
print(f"Tahmin olasılıkları tablosu kaydedildi: {probs_table_path}")

# --- C) Feature-Value Tablosu (En önemli özelliklerin gerçek değerleri) ---

print("   -> Feature-Value tablosu oluşturuluyor...")

# DÜZELTME: as_list yerine as_map kullanıyoruz.
# as_map bize [(feature_index, weight), ...] formatında döner.
lime_map = exp.as_map()[pred_label_idx]

# Sadece ilk 10 özelliğin indeksini al (LIME zaten sıralı verir)
top_features_indices = [x[0] for x in lime_map[:10]]

# Bu indeksleri kullanarak isimleri çek
top_features_names = [FEATURE_NAMES[i] for i in top_features_indices]

# Bu indeksleri kullanarak orijinal değerleri çek
top_features_values = [patient_data_original[i] for i in top_features_indices]

feat_val_df = pd.DataFrame({
    'Özellik (Feature)': top_features_names, 
    'Değer (Value)': np.round(top_features_values, 4)
})

fig, ax = plt.subplots(figsize=(8, 6))
ax.axis('tight')
ax.axis('off')
# Tabloyu çiz
table = ax.table(cellText=feat_val_df.values, colLabels=feat_val_df.columns, loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.2)
plt.title(f"Hasta {patient_idx} İçin En Önemli 10 Özellik ve Değerleri", fontsize=14, fontweight='bold')

feat_val_table_path = os.path.join(RESULTS_PATH, f'lime_feat_val_table_hasta_{patient_idx}.png')
plt.savefig(feat_val_table_path, bbox_inches='tight', dpi=150)
plt.close()
print(f"Feature-Value tablosu kaydedildi: {feat_val_table_path}")


--- LIME ANALİZİ BAŞLATILIYOR ---
   -> Analiz edilen hasta indeksi: 5082
LIME grafiği kaydedildi: /media/data/bitirme son/bitirme-projesi/results/lime_analizi_hasta_5082.png
   -> Tahmin olasılıkları tablosu oluşturuluyor...
Tahmin olasılıkları tablosu kaydedildi: /media/data/bitirme son/bitirme-projesi/results/lime_probs_table_hasta_5082.png
   -> Feature-Value tablosu oluşturuluyor...
Feature-Value tablosu kaydedildi: /media/data/bitirme son/bitirme-projesi/results/lime_feat_val_table_hasta_5082.png


In [17]:
# 3. GRAD-CAM ANALİZİ (Görsel Açiklama)
# =============================================================================
print("\n--- GRAD-CAM ANALİZİ BAŞLATILIYOR ---")

def calculate_gradcam_heatmap(input_image, cnn_model, target_layer_name, pred_index=None):
    grad_model = Model(
        inputs=cnn_model.inputs,
        outputs=[cnn_model.get_layer(target_layer_name).output, cnn_model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(input_image)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1))
    
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

# Hedef katmanı bul
target_layer = ""
for layer in reversed(resnet_model.layers):
    if 'conv1d' in layer.name:
        target_layer = layer.name
        break

if target_layer == "":
    print("⚠️ Modelde uygun bir Conv1D katmani bulunamadi. Grad-CAM atlanacak.")
else:
    random_sig_idx = np.random.randint(0, X_test_sig.shape[0])
    print(f"   -> Analiz edilen sinyal indeksi: {random_sig_idx}")

    t_steps = X_test_sig.shape[1]
    n_ch = X_test_sig.shape[2]
    sample_signal = X_test_sig[random_sig_idx].reshape(1, t_steps, n_ch)

    predictions = resnet_model.predict(sample_signal)
    predicted_class_idx = np.argmax(predictions)
    predicted_label = CLASS_NAMES[predicted_class_idx]

    true_label_idx = y_test[random_sig_idx]
    true_label = CLASS_NAMES[true_label_idx]

    heatmap_result = calculate_gradcam_heatmap(sample_signal, resnet_model, target_layer)

    if heatmap_result is not None:
        plt.figure(figsize=(15, 6))
        
        # Sinyalin sadece bir lead'ini (örn: Lead I) çizelim ki anlaşılır olsun
        signal_to_plot = sample_signal[0, :, 0]
        
        zoom_factor = len(signal_to_plot) / len(heatmap_result)
        heatmap_resized = scipy.ndimage.zoom(heatmap_result, zoom_factor)
        
        plt.plot(signal_to_plot, 'k', linewidth=1.5, label='EKG Sinyali (Lead I)')
        
        plt.imshow(
            [heatmap_resized], 
            aspect='auto', 
            cmap='jet', 
            alpha=0.6, 
            extent=[0, len(signal_to_plot), min(signal_to_plot), max(signal_to_plot)]
        )
        
        plt.colorbar(label="Model Odagi (Aktivasyon)")
        # Başlığa Gerçek Etiketi de ekledim
        plt.title(f"Grad-CAM (Hasta: {random_sig_idx} | Gerçek: {true_label} | Tahmin: {predicted_label})")
        plt.xlabel("Zaman Adimlari (Time Steps)")
        plt.ylabel("Genlik (mV)")
        plt.legend()
        
        gradcam_save_path = os.path.join(RESULTS_PATH, f'gradcam_hasta_{random_sig_idx}.png')
        plt.savefig(gradcam_save_path, bbox_inches='tight', dpi=150)
        plt.close()
        print(f"Grad-CAM grafiği kaydedildi: {gradcam_save_path}")

print("\nTÜM ANALİZLER BAŞARIYLA TAMAMLANDI VE KAYDEDİLDİ.")


--- GRAD-CAM ANALİZİ BAŞLATILIYOR ---
   -> Analiz edilen sinyal indeksi: 6795
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 534ms/step
Grad-CAM grafiği kaydedildi: /media/data/bitirme son/bitirme-projesi/results/gradcam_hasta_6795.png

TÜM ANALİZLER BAŞARIYLA TAMAMLANDI VE KAYDEDİLDİ.
